# 🎙️ Préparation et Clonage d'une Voix Narrateur pour Pocket-TTS (NekoSpeak)

Ce notebook vous permet de créer une voix haute fidélité pour **Pocket-TTS** à partir de vos enregistrements audio (ex: Thierry Blanc / Narrateur), de **tester la synthèse directement dans Colab** en français, et d'**exporter les fichiers prêts pour NekoSpeak** (soit via HuggingFace, soit directement dans l'application).

### ⚡ Pourquoi Pocket-TTS plutôt qu'un fine-tune LoRA (OmniVoice) ou Kokoro ?
- **Zéro entraînement lourd** : Pocket-TTS utilise le codec **Mimi (Kyutai)** pour extraire instantanément l'empreinte vocale (timbre, grain, résonance) à partir de 15–30 secondes d'audio.
- **Spécialiste du français** : Le modèle `french_24l` est entraîné sur le corpus français **Expresso** (locuteurs natifs parisiens), garantissant une diction et des liaisons naturelles.
- **Vitesse sur S25 Ultra** : Modèle INT8 de ~150 Mo, génération en **1 à 2 secondes par phrase** sur smartphone (contre 60 à 75 secondes pour OmniVoice).

## 1. Installation des dépendances
On installe les outils audio (`soundfile`, `librosa`, `scipy`) et le moteur Pocket-TTS.

In [ ]:
!pip install -q soundfile librosa scipy huggingface_hub onnxruntime numpy

## 2. Configuration & Connexion Google Drive (Optionnel)
Si vos enregistrements audio se trouvent sur Google Drive, montez votre Drive ci-dessous.

In [ ]:
import os
from google.colab import drive

# Monter le drive si vous avez vos fichiers dans 'Mon Drive/thierry_blanc/'
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Répertoire de travail
WORK_DIR = '/content/work_pocket'
os.makedirs(WORK_DIR, exist_ok=True)

# Identifiant et nom de la voix pour NekoSpeak
VOICE_ID = 'narrateur_thierry'
VOICE_DISPLAY_NAME = 'Thierry Blanc (Narrateur)'

## 3. Sélection et Nettoyage de l'Audio de Référence

Pour Pocket-TTS, la qualité du clone dépend principalement de la **propreté** de l'extrait de référence :
- Durée idéale : **15 à 30 secondes** de parole continue.
- **Pas de musique de fond**, pas d'écho de pièce prononcé.
- Le notebook applique un filtre passe-haut (80 Hz) pour supprimer les ronflements de micro, coupe les silences et normalise l'amplitude.

In [ ]:
import soundfile as sf
import librosa
import numpy as np
from scipy import signal
import IPython.display as ipd

# Remplacez par le chemin vers votre fichier audio source (WAV ou MP3)
# Par exemple : '/content/drive/MyDrive/thierry_blanc/extrait.wav'
SOURCE_AUDIO_PATH = '/content/drive/MyDrive/thierry_blanc/extrait.wav'

# Si vous n'avez pas encore mis le fichier dans Drive, vous pouvez le téléverser directement dans Colab :
if not os.path.exists(SOURCE_AUDIO_PATH):
    print(f"⚠️ Fichier non trouvé à : {SOURCE_AUDIO_PATH}")
    print("Téléversez votre fichier audio dans Colab et modifiez SOURCE_AUDIO_PATH.")
else:
    print(f"Fichier trouvé : {SOURCE_AUDIO_PATH}")

def prepare_reference_audio(input_path, output_path, target_sr=24000, max_seconds=30.0):
    """
    Prépare l'audio de référence selon le pipeline standard de NekoSpeak :
    1. Conversion en mono 24000 Hz
    2. Filtre passe-haut 80 Hz (suppression grondements)
    3. Découpe des silences aux extrémités
    4. Normalisation de l'amplitude (-1.0 à +1.0)
    5. Troncature aux 30 premières secondes
    """
    # Chargement en mono à 24 kHz
    y, sr = librosa.load(input_path, sr=target_sr, mono=True)
    
    # Filtre passe-haut 80 Hz (Butterworth ordre 2)
    sos = signal.butter(2, 80, btype='highpass', fs=sr, output='sos')
    y_filtered = signal.sosfilt(sos, y)
    
    # Découpe des silences (seuil à 30 dB sous le pic)
    y_trimmed, _ = librosa.effects.trim(y_filtered, top_db=30)
    
    # Troncature max_seconds
    max_samples = int(max_seconds * sr)
    if len(y_trimmed) > max_samples:
        y_trimmed = y_trimmed[:max_samples]
        
    # Normalisation Peak
    peak = np.max(np.abs(y_trimmed))
    if peak > 0:
        y_normalized = y_trimmed / peak * 0.95
    else:
        y_normalized = y_trimmed
        
    # Sauvegarde en WAV PCM 16-bit 24kHz
    sf.write(output_path, y_normalized, sr, subtype='PCM_16')
    print(f"✅ Audio préparé : {output_path} ({len(y_normalized)/sr:.1f}s, {sr} Hz)")
    return y_normalized, sr

CLEAN_WAV_PATH = os.path.join(WORK_DIR, f'{VOICE_ID}.wav')

if os.path.exists(SOURCE_AUDIO_PATH):
    audio_ref, sr = prepare_reference_audio(SOURCE_AUDIO_PATH, CLEAN_WAV_PATH)
    ipd.display(ipd.Audio(CLEAN_WAV_PATH))


## 4. Téléchargement des Modèles Pocket-TTS Français (ONNX)
On télécharge le bundle officiel français `french_24l` de `KevinAHM/pocket-tts-onnx` utilisé par NekoSpeak.

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "KevinAHM/pocket-tts-onnx"
BUNDLE = "onnx/french_24l"
MODELS_DIR = os.path.join(WORK_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

FILES = [
    "mimi_encoder_int8.onnx",
    "text_conditioner_int8.onnx",
    "flow_lm_main_int8.onnx",
    "flow_lm_flow_int8.onnx",
    "mimi_decoder_int8.onnx",
    "tokenizer.model",
    "bos_before_voice.npy"
]

downloaded_files = {}
for f in FILES:
    path = hf_hub_download(repo_id=REPO_ID, filename=f"{BUNDLE}/{f}", local_dir=WORK_DIR)
    downloaded_files[f] = path
    print(f"Téléchargé : {f}")

## 5. Extraction de l'Empreinte Vocale (Mimi Codec)
L'encodeur `mimi_encoder` compresse les 15–30s de l'audio de référence en une matrice de latents acoustiques de dimension 1024.
Cette étape prend moins de **0.5 seconde** !

In [ ]:
import onnxruntime as ort

mimi_encoder_path = downloaded_files["mimi_encoder_int8.onnx"]
session_encoder = ort.InferenceSession(mimi_encoder_path, providers=['CPUExecutionProvider'])

# Lecture de l'audio propre à 24kHz
wav_samples, sr = sf.read(CLEAN_WAV_PATH)
if wav_samples.ndim > 1:
    wav_samples = wav_samples.mean(axis=1)

# Préparation du tenseur [1, 1, num_samples]
audio_tensor = wav_samples.astype(np.float32).reshape(1, 1, -1)

# Inférence mimi_encoder
outputs = session_encoder.run(None, {'audio': audio_tensor})
voice_latents = outputs[0]  # Forme [1, num_frames, 1024]
num_frames = voice_latents.shape[1]

print(f"✅ Empreinte vocale extraite avec succès !")
print(f"   Frames capturées : {num_frames} frames (environ {num_frames / 12.5:.1f}s de représentation vocale)")
print(f"   Dimension par frame : {voice_latents.shape[2]} (latents Mimi)")

## 6. Génération et Test d'Écoute (TTS Complet)
Testons immédiatement la voix synthétisée sur un texte narratif en français !

In [ ]:
import struct

# Texte de test pour le narrateur
TEXT_TEST = "Il déambula dans les rues désertes et les jardins publics d'Agrillon, réfléchissant avec gravité aux propos de l'estropier."

print(f"Texte à tester :\n« {TEXT_TEST} »\n")

# Vous pouvez installer le package pocket-tts de Kyutai pour une inférence Python ultra simple avec votre audio :
!pip install -q pocket-tts

from pocket_tts import PocketTTS
import torch

# Initialisation du modèle français officiel
tts = PocketTTS(language="fr")

# Génération avec votre audio de référence propre !
audio_out = tts.generate(text=TEXT_TEST, voice=CLEAN_WAV_PATH)

# Sauvegarde du résultat de test
TEST_OUT_PATH = os.path.join(WORK_DIR, 'test_synthese.wav')
sf.write(TEST_OUT_PATH, audio_out, 24000)
print(f"✅ Synthèse réussie : {TEST_OUT_PATH}")

ipd.display(ipd.Audio(TEST_OUT_PATH))

## 7. Export des Fichiers pour NekoSpeak

NekoSpeak supporte plusieurs manières d'importer votre voix :

### Option A : Le fichier WAV propre (`.wav`)
- Vous pouvez l'importer directement dans NekoSpeak via le bouton **« Cloner une voix »** dans l'onglet Voix.
- Ou le mettre sur **Hugging Face** pour qu'il soit téléchargeable automatiquement comme voix officielle.

### Option B : Le fichier d'état binaire NekoSpeak (`.bin` - format `PocketVoiceState` v1)
- Ce fichier contient l'identifiant, le nom d'affichage et les latents déjà encodés.
- Si vous le déposez dans le dossier `pocket/cloned_voices/` de l'application, la voix apparaît immédiatement sans avoir besoin d'être ré-encodée.

### Option C : Le fichier cache d'embeddings (`.emb`)
- Utilisé par NekoSpeak pour le cache interne sous `pocket/voice_cache/<id>.emb`.

In [ ]:
# 1. Export du fichier cache .emb (format: [num_frames: int32, latents: float32...])
emb_path = os.path.join(WORK_DIR, f"{VOICE_ID}.emb")
flattened_latents = voice_latents.flatten().astype(np.float32)
with open(emb_path, 'wb') as f:
    f.write(struct.pack('<i', num_frames))
    f.write(flattened_latents.tobytes())
print(f"Exporté : {emb_path} ({os.path.getsize(emb_path)/1024:.1f} Ko)")

# 2. Export du fichier binaire PocketVoiceState v1 .bin (Magic 'PKVS')
# Format: MAGIC(4B='PKVS') + VERSION(2B=1) + RESERVED(2B=0) + ID_LEN(4B) + ID + NAME_LEN(4B) + NAME + NUM_FRAMES(4B) + IS_BUNDLED(1B) + CREATED_AT(8B) + LATENTS
bin_path = os.path.join(WORK_DIR, f"{VOICE_ID}.bin")
id_bytes = VOICE_ID.encode('utf-8')
name_bytes = VOICE_DISPLAY_NAME.encode('utf-8')
import time
created_at = int(time.time() * 1000)

with open(bin_path, 'wb') as f:
    f.write(struct.pack('<I', 0x53564B50))       # MAGIC 'PKVS' en little-endian
    f.write(struct.pack('<h', 1))                # VERSION 1
    f.write(struct.pack('<h', 0))                # Reserved
    f.write(struct.pack('<i', len(id_bytes)))    # id length
    f.write(id_bytes)                            # id
    f.write(struct.pack('<i', len(name_bytes)))  # name length
    f.write(name_bytes)                          # name
    f.write(struct.pack('<i', num_frames))       # numFrames
    f.write(struct.pack('<b', 0))                # isBundled (0 = cloned, 1 = bundled)
    f.write(struct.pack('<q', created_at))       # createdAt (timestamp ms)
    f.write(flattened_latents.tobytes())         # latents

print(f"Exporté : {bin_path} ({os.path.getsize(bin_path)/1024:.1f} Ko)")

# Création d'une archive ZIP prête au téléchargement
import shutil
ZIP_OUTPUT = os.path.join('/content', f'voix_pocket_{VOICE_ID}.zip')
export_folder = os.path.join(WORK_DIR, 'export')
os.makedirs(export_folder, exist_ok=True)

shutil.copy(CLEAN_WAV_PATH, os.path.join(export_folder, f'{VOICE_ID}.wav'))
shutil.copy(bin_path, os.path.join(export_folder, f'{VOICE_ID}.bin'))
shutil.copy(emb_path, os.path.join(export_folder, f'{VOICE_ID}.emb'))

shutil.make_archive(ZIP_OUTPUT.replace('.zip', ''), 'zip', export_folder)
print(f"\n🎉 Tout est prêt ! Archive téléchargeable : {ZIP_OUTPUT}")

## 8. Comment intégrer cette voix dans NekoSpeak ?

### Méthode 1 (Directe sur votre smartphone)
1. Téléchargez le fichier `{VOICE_ID}.wav` sur votre smartphone.
2. Dans NekoSpeak, allez dans **Voix** → **Clonage de voix**.
3. Cliquez sur **« Importer un audio »**, sélectionnez votre WAV, donnez-lui un nom.
4. C'est tout ! Votre voix fonctionne immédiatement.

### Méthode 2 (Téléchargeable via Hugging Face ou en dur)
1. Uploadez `{VOICE_ID}.wav` sur votre dépôt Hugging Face.
2. Dans NekoSpeak (`VoiceDefinitions.kt`), ajoutez la voix dans `POCKET_VOICES` :
```kotlin
VoiceDefinition(
    id = "narrateur_thierry",
    name = "Thierry Blanc (Narrateur)",
    gender = "Male",
    region = "French",
    description = "Narration dark fantasy grave et posée",
    sampleUrl = "https://huggingface.co/<votre_repo>/resolve/main/narrateur_thierry.wav",
    modelType = "pocket_v1"
)
```
3. Dans `ModelRepository.kt`, ajoutez le fichier sous `pocket_v1` :
```kotlin
ModelFile("pocket/voices/narrateur_thierry.wav", "https://huggingface.co/<votre_repo>/resolve/main/narrateur_thierry.wav", "Thierry Blanc (Narrateur)")
```
L'application téléchargera automatiquement le fichier WAV et générera le cache au premier lancement !